# UQRoute-TC — Task 2 saved-pilot audit (CPU only)

Run the three code cells in order in a Colab CPU runtime. The notebook reads the four completed five-case pilots from Drive, checks the pinned benchmark parser and token evidence, and saves a compact audit JSON in Drive. It does not serve a model or run held-out cases.


## 1. Restore pinned benchmark and current audit code

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import sys

drive.mount('/content/drive')
pilot_root = Path('/content/drive/MyDrive/UQRoute-TC/pilots')
assert pilot_root.is_dir(), 'UQRoute-TC/pilots folder is missing in Drive'
benchmark = Path('/content/robustbench-tc-release')
if not benchmark.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WillChow66/robustbench-tc-release.git', str(benchmark)], check=True)
benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
subprocess.run(['git', '-C', str(benchmark), 'checkout', '--detach', benchmark_revision], check=True)
assert subprocess.check_output(['git', '-C', str(benchmark), 'rev-parse', 'HEAD'], text=True).strip() == benchmark_revision
project = Path('/content/uqroute-tc-task2-audit')
branch = 'feat/task2-canonical-parser'
if not project.exists():
    subprocess.run(['git', 'clone', '--branch', branch, '--single-branch',
                    'https://github.com/RavindraSSK/uqroute-tc.git', str(project)], check=True)
else:
    subprocess.run(['git', '-C', str(project), 'fetch', 'origin', branch], check=True)
    subprocess.run(['git', '-C', str(project), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
assert (project / 'src/uqroute_tc/parsing/audit.py').is_file(), 'Audit code missing; fetch latest branch'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{project}[client]'], check=True)
print('Audit code revision:', subprocess.check_output(['git', '-C', str(project), 'rev-parse', 'HEAD'], text=True).strip())
print('Benchmark revision:', benchmark_revision)


## 2. Check the 20 saved records (no inference)

In [ ]:
import subprocess
import sys

audit_path = pilot_root / 'task2_saved_pilot_audit_v1.json'
subprocess.run([sys.executable, '-m', 'uqroute_tc.parsing.audit',
                '--benchmark-root', str(benchmark),
                '--pilot-root', str(pilot_root),
                '--project-root', str(project),
                '--output', str(audit_path)], check=True)


## 3. Show representative raw output and parsed-call status for review

In [ ]:
import json

report = json.loads(audit_path.read_text())
print('Records:', report['total'])
print('Parser statuses:', report['parser_status_counts'])
print('Matching saved calls:', report['parser_matches_saved'])
print('Valid single-sample scores:', report['valid_single_sample_scores'])
for model in dict.fromkeys(entry['model_id'] for entry in report['entries']):
    example = next(entry for entry in report['entries'] if entry['model_id'] == model)
    print('\nMODEL:', model)
    print('Case:', example['case_id'], '| parsed calls:', example['reparsed_call_count'])
    print('Raw preview:', repr(example['raw_preview']))
print('\nSaved:', audit_path)
